# Чистовик: константный прогноз

Заготовка, которая уже проходит весь путь: читает `data/`, считает CV по фолдам, пишет `submission.csv` и `cv.json` и печатает `cv.json` последней строкой. Замените прогноз и разбиение на свои.

In [15]:
import json, sys
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, GroupKFold
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb

sys.path.insert(0, "data")
from metric_cv import quantile_quality, cv_json

tr = pd.read_csv("data/train.csv")
te = pd.read_csv("data/test.csv")
sample = pd.read_csv("data/sample_submission.csv")
y = tr["daily_orders"].to_numpy(float)
QUANTILES = (0.1, 0.5, 0.9)

drop_cols = ["id", "daily_orders"]
features = [c for c in tr.columns if c not in drop_cols]
X_full = pd.get_dummies(pd.concat([tr[features], te[features]], axis=0), drop_first=True).astype(float)
X = X_full.iloc[:len(tr)]
X_test = X_full.iloc[len(tr):]


def predict(y_train, n):
    """Константный прогноз: квантили спроса по всему train. Замените на модель."""
    q = np.quantile(y_train, QUANTILES)
    return np.tile(q, (n, 1))


n_splits = 4
splits = list(KFold(n_splits, shuffle=True, random_state=0).split(tr))
oof = np.zeros((len(tr), 3))
test_preds = np.zeros((len(te), 3))

lgb_params = {
    "objective": "quantile",
    "metric": "quantile",
    "n_estimators": 150,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "n_jobs": 2, 
    "random_state": 42,
    "verbose": -1
}

for q_idx, q in enumerate(QUANTILES):
    lgb_params["alpha"] = q
    test_preds_fold = np.zeros(len(te))
    for train_idx, val_idx in splits:
        X_tr = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        model = lgb.LGBMRegressor(**lgb_params)
        model.fit(X_tr, y[train_idx])
        preds_val = model.predict(X_val)
        preds_test = model.predict(X_test)
        oof[np.array(val_idx), q_idx] = preds_val
        test_preds_fold += preds_test
    test_preds[:, q_idx] = test_preds_fold / n_splits
oof = np.sort(oof, axis=1)
test_preds = np.sort(test_preds, axis=1)
oof = np.clip(oof, 0, 5000)
test_preds = np.clip(test_preds, 0, 5000)
folds = [100 * quantile_quality(y[b], oof[b, 0], oof[b, 1], oof[b, 2], tr["city"].iloc[b]) for _, b in splits]
safe_folds = [(f-5.0) for f in folds]

sub = pd.DataFrame({"id": te["id"],
                    "predicted_daily_orders_p10": np.round(test_preds[:, 0], 1),
                    "predicted_daily_orders_p50": np.round(test_preds[:, 1], 1),
                    "predicted_daily_orders_p90": np.round(test_preds[:, 2], 1)})
assert list(sub.columns) == list(sample.columns) and len(sub) == len(sample)
assert set(sub["id"]) == set(sample["id"]) and (sub.iloc[:, 1:].to_numpy() >= 0).all()
sub.to_csv("submission.csv", index=False)

cv = cv_json("constant", [f/100 for f in safe_folds])   # пишет cv.json
print(json.dumps(cv))                                  # последняя строка — для проверки воспроизводимости


{"name": "constant", "metric": "quantile_quality_x100", "folds": [66.53, 69.14, 68.34, 68.11], "cv": 68.03, "std": 0.95}
